Source paper: PI-Link: A Ground-Truth Dataset of Links Between Pull-Requests and Issues in GitHub

paper link: https://ieeexplore.ieee.org/abstract/document/10002372

dataset link: https://www.kaggle.com/datasets/zakareaalshara/android-closed-issues-20110101-20210101-clean

The dataset contains links between pull requests and issues collected from Android projects on GitHub. It is available on kaggle in the form of one JSON file. The code in "json_processing.py" transforms it into 3 dataframes: Pull_Requests, Issues and links between them.

In [16]:
import kagglehub

from json_processing import load_json_dataframes, construct_issue_links, normalise_labels

In [17]:
path = kagglehub.dataset_download("zakareaalshara/android-closed-issues-20110101-20210101-clean")

print("Path to dataset files:", path)

(issues, prs, links) = load_json_dataframes(path)
issue_links = construct_issue_links(issues, links)

Path to dataset files: C:\Users\dhali\.cache\kagglehub\datasets\zakareaalshara\android-closed-issues-20110101-20210101-clean\versions\3
Total JSON entries: 34732


In [18]:
issues.duplicated(subset="issue_title").sum()
issues.duplicated().sum()

np.int64(0)

In [19]:
issues[issues.duplicated(subset=["issue_url"],keep=False)]

,issue_no,issue_key,issue_url,issue_title,issue_body,repository,created_at,closed_at,labels,comments


Issues table:

In [20]:
print(f"Nr of issues: {len(issues)}")
print(f"Columns: {issues.columns.values.tolist()}")

Nr of issues: 27232
Columns: ['issue_no', 'issue_key', 'issue_url', 'issue_title', 'issue_body', 'repository', 'created_at', 'closed_at', 'labels', 'comments']


In [21]:
first_row = issues.iloc[0]
print(f"First issue nr: {first_row['issue_no']}")
print(f"url: {first_row['issue_url']}")
print(f"title: {first_row['issue_title']}")

First issue nr: 14
url: https://github.com/05nelsonm/TorOnionProxyLibrary-Android/issues/14
title: App crashing sometimes when ServiceAction.RESTART is being executed.


In [22]:
print(f"body: {first_row['issue_body']}")

body: # Description
<!-- Short description of the issue  -->
App is crashing after Tor is restarted. It reaches 95% Bootstrapped, stalls, then crashes.

## Steps To Reproduce
<!--- Provide a link to a live example, or an unambiguous set of steps to -->
<!--- reproduce this bug. Include code to reproduce, if relevant          -->
Happens sporadically, and unsure if it's only something with sending the `ServiceAction` via the notification's `PendingIntent`, or if it's also happening with the `TorServiceController.restartTor` method, too.


In [23]:
print(f"repository: {first_row['repository']}")

repository: 05nelsonm/TorOnionProxyLibrary-Android


In [24]:
print(f"created at: {first_row['created_at']}")
print(f"closed at: {first_row['closed_at']}")

created at: 2020-06-26T17:24:15Z
closed at: 2020-07-02T21:27:15Z


In [25]:
print(f"labels: {first_row['labels']}")

labels: bug


In [26]:
# print(f"comments:\n {first_row['comments']}")
# issues["comments"].isnull().sum()
issues["comments"]

0        Stack trace:\r\n```\r\n2020-06-28 06:05:32.947...
2        Hmm. Right now the service is being started by...
3        Ok, the Nr Format is +<countycode><number with...
4        The comments in this [pyenv issue](https://git...
5        You mean that on previous release you xere abl...
                               ...                        
34727    Please copy and paste your error log. | I seem...
34728    I'm almost sure that #176 fixes this issue. | ...
34729    Got this same issue on android did you resolve...
34730                     I am having the same exact error
34731    能再给一些信息吗？ | 之前这功能是正常的\r\n\r\n随便进一个主题列表，滑到底部，上拉...
Name: comments, Length: 27232, dtype: str

Linked_issues is a table assigning to each issue the issues that share at least one pr with it. It is the table storing the ground truth for finding connections between issues. It was constructed based on the data frames: prs and links.

In [27]:
print(f"Nr of entries: {len(issue_links)}")
print(f"Columns: {issue_links.columns.values.tolist()}")

Nr of entries: 27232
Columns: ['issue_key', 'related_issue_keys']


In [28]:
print(issue_links)

                                       issue_key related_issue_keys
0      05nelsonm/TorOnionProxyLibrary-Android#14                 []
1      05nelsonm/TorOnionProxyLibrary-Android#33                 []
2                     0x5ECF4ULT/PanicTrigger#31                 []
3                                  18F/laptop#52                 []
4               1technophile/OpenMQTTGateway#101                 []
...                                          ...                ...
27227                   zxing-js/ngx-scanner#164                 []
27228                zyra/ionic-image-loader#174                 []
27229                zyra/ionic-image-loader#183                 []
27230                  zyra/ionic-super-tabs#203                 []
27231                    zzq889/stage1st-app#109                 []

[27232 rows x 2 columns]


Currently the issue is entered into issue_links only if it has documented link to some pr, which is why there are significantly less entries in this table comparing to Issues table.

## Label normalisation

In the raw dataset there are a lot of redundant labels, as can be seen below.

In [29]:
from collections import Counter

label_counts = Counter(
    label
    for labels in issues["labels"].dropna()
    for label in labels.split("|")
    if label
)

for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
    print(f"{label}: {count}")

bug: 5258
enhancement: 1816
Bug: 1074
Android: 688
good first issue: 613
help wanted: 543
in progress: 494
[QA]:Verified fixed: 469
android: 426
[Type] Bug: 422
type: bug: 373
feature: 307
pr exists: 289
[QA]:Normal issue: 235
regression: 231
Resolution: Locked: 213
approved: 212
Type: Bug: 202
framework: 183
[Type] Enhancement: 170
released: 168
翻译完成: 160
state: verified fixed: 157
question: 154
triaged: 150
🐞 bug: 148
t/bug :bug:: 148
UI: 145
type: bug/fix: 138
[Type] Crash: 131
hacktoberfest: 127
P1: 126
iOS: 126
Enhancement: 125
Status: Not started: 114
[QA]:Major issue: 107
high priority: 104
wallet: 102
welcome contribute: 100
assigned: 99
tool: 99
release: 96
os: android: 93
high-priority: 91
documentation: 90
severe: crash: 90
eng:qa:verified: 90
[Pri] High: 87
Core: 85
Hacktoberfest: 85
P2: 84
f: material design: 84
Reader: 82
p/Android: 81
defect: 79
bounty: 79
plugin: 78
Type: Confirmed bug: 76
up-for-grabs: 76
MVP: 76
[QA]:Caught_by_exploratory: 76
type: crash: 76
Posting/E

The raw `labels` field contains hundreds of free-text variants across repositories (e.g. `"type: bug"`, `"[Type] Bug"`, `"🐞 Bug"` all mean the same thing). To make labels usable for analysis and modelling, they are mapped to a small set of canonical group names organised into 6 dimensions:

| Dimension | Values | Example raw labels |
|---|---|---|
| **type** | `bug`, `crash`, `regression`, `enhancement`, `task`, `question`, `documentation` | `type: bug`, `[Type] Bug`, `feature request` |
| **priority** | `priority:critical`, `priority:high`, `priority:medium`, `priority:low` | `P1`, `HIGH PRIORITY`, `low-severity` |
| **status** | `status:in_progress`, `status:qa`, `status:resolved`, `status:stale` | `in progress`, `QA-verified`, `wontfix` |
| **platform** | `platform:android`, `platform:ios`, `platform:web`, `platform:linux` | `OS: Android`, `p/iOS 🍎` |
| **layer** | `layer:frontend`, `layer:backend`, `layer:infrastructure` | `UI`, `ux`, `backend`, `API`, `Build` |
| **community** | `community:good_first_issue`, `community:help_wanted`, `community:bounty` | `good first issue`, `Hacktoberfest` |

Each issue's `labels` field is replaced with a pipe-separated string of matched canonical names (e.g. `"bug|layer:frontend|priority:high"`). Labels that do not match any group are dropped. The normalisation is implemented in `json_processing.normalise_labels()`.

In [ ]:
issues = normalise_labels(issues)
label_counts = Counter(
    label
    for labels in issues["labels"].dropna()
    for label in labels.split("|")
    if label
)

for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
    print(f"{label}: {count}")

bug: 6780
enhancement: 2755
status:in_progress: 1204
platform:android: 1008
status:qa: 866
community:good_first_issue: 848
priority:high: 663
layer:frontend: 546
community:help_wanted: 543
status:stale: 436
priority:medium: 414
status:resolved: 394
crash: 288
priority:low: 268
regression: 262
platform:ios: 240
community:bounty: 223
question: 210
documentation: 171
priority:critical: 160
task: 119
layer:backend: 103
layer:infrastructure: 77
platform:web: 22
platform:linux: 22


In [30]:
issues["comments"]

0        Stack trace:\r\n```\r\n2020-06-28 06:05:32.947...
2        Hmm. Right now the service is being started by...
3        Ok, the Nr Format is +<countycode><number with...
4        The comments in this [pyenv issue](https://git...
5        You mean that on previous release you xere abl...
                               ...                        
34727    Please copy and paste your error log. | I seem...
34728    I'm almost sure that #176 fixes this issue. | ...
34729    Got this same issue on android did you resolve...
34730                     I am having the same exact error
34731    能再给一些信息吗？ | 之前这功能是正常的\r\n\r\n随便进一个主题列表，滑到底部，上拉...
Name: comments, Length: 27232, dtype: str

In [ ]:
issues.to_csv("issues.csv", index=False)
print("Saved issues.csv")

Saved issues.csv
